# 06 · Evaluating AI Systems

> **Source notes:** `EvaluatingAISystems.md`

ML metrics work when you have labels. AI systems produce free-form text — evaluation requires different approaches:
- **Level 1 — Component eval**: retrieval precision/recall
- **Level 2 — Pipeline eval**: RAGAS-style faithfulness, relevance, context precision
- **Level 3 (proxy) — LLM-as-judge**: pairwise preference with position-bias check
- **Regression harness**: lock in thresholds before shipping

Uses Ollama (local) and `sentence-transformers`. No API key needed.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ollama','sentence-transformers','-q'],check=True)
print('Packages ready.')
import ollama, json, re, numpy as np
from sentence_transformers import SentenceTransformer, util
MODEL = 'phi3:mini'
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
def llm(system, user, temperature=0.0, max_tokens=200):
    r = ollama.chat(model=MODEL,messages=[{'role':'system','content':system},{'role':'user','content':user}],options={'temperature':temperature,'num_predict':max_tokens})
    return r['message']['content'].strip()
def embed(texts): return embed_model.encode(texts, normalize_embeddings=True)
print('Models loaded.')

## 1 · Level 1 — Retrieval Evaluation (Precision@k, Recall@k)

In [ ]:
KB = {
    'menu-01': 'Margherita: 9.99. GF base +1.50. Contains dairy and gluten.',
    'menu-02': 'Pepperoni: 11.99. Contains dairy, gluten, and pork.',
    'menu-03': 'Veggie Supreme: 10.49. Contains dairy and gluten. No meat.',
    'menu-04': 'Garlic Bread: 3.49. Contains dairy and gluten.',
    'ops-01':  'Delivery: central (all week), north (Mon-Fri). Min order 15.',
    'ops-02':  'Hours: Mon-Thu 11am-10pm, Fri-Sat 11am-11pm, Sun noon-9pm.',
    'ops-03':  'Allergen policy: cross-contamination possible. Not safe for severe allergies.',
    'ops-04':  'Payment: card and cash. Online orders: card only.',
}
EVAL_DATA = [
    ('What gluten-free options?',           {'menu-01'}),
    ('Does Pepperoni contain pork?',        {'menu-02'}),
    ('Delivery on Monday evening?',         {'ops-01'}),
    ('Allergens in Margherita?',            {'menu-01','ops-03'}),
    ('Accept cash payments?',               {'ops-04'}),
]
chunk_ids = list(KB.keys()); chunk_texts = list(KB.values())
chunk_embs = embed(chunk_texts)
def retrieve(query, k=3):
    q_emb = embed([query])
    scores = util.cos_sim(q_emb, chunk_embs)[0].numpy()
    return [chunk_ids[i] for i in np.argsort(scores)[::-1][:k]]
K = 3
prec_scores, rec_scores = [], []
print(f'{"Query":<48} {"P@k":>6} {"R@k":>6}')
print('-'*65)
for q, rel in EVAL_DATA:
    ret  = retrieve(q, K)
    hits = len(set(ret) & rel)
    p    = hits/K; r = hits/len(rel)
    prec_scores.append(p); rec_scores.append(r)
    short = q[:45]+'...' if len(q)>45 else q
    print(f'{short:<48} {p:>6.2f} {r:>6.2f}')
print(f'\nMean P@{K}: {np.mean(prec_scores):.3f}   Mean R@{K}: {np.mean(rec_scores):.3f}')

## 2 · Level 2 — RAGAS-Style Pipeline Metrics

| Metric | What it measures |
|---|---|
| **Faithfulness** | Claims in answer supported by context? |
| **Answer Relevance** | Answer addresses the question? |
| **Context Precision** | Retrieved chunks actually relevant? |

In [ ]:
def score_faithfulness(answer, context):
    ext = llm('Extract every factual claim as JSON list of strings. Only JSON.', f'Answer: {answer}', max_tokens=200)
    try:
        m = re.search(r'\[.*\]', ext, re.DOTALL)
        claims = json.loads(m.group()) if m else [answer]
    except: claims = [answer]
    if not claims: return 1.0
    sys = 'Is this claim supported by the context? Reply yes or no only.'
    supported = sum(1 for c in claims if 'yes' in llm(sys, f'Context: {context}\nClaim: {c}', max_tokens=5).lower())
    return supported / len(claims)

def score_answer_relevance(question, answer):
    return float(util.cos_sim(embed([question]), embed([answer]))[0][0])

def score_context_precision(question, contexts, k=3):
    sys = 'Is this context relevant to the question? Reply yes or no only.'
    rel = sum(1 for ctx in contexts[:k] if 'yes' in llm(sys, f'Q: {question}\nCtx: {ctx}', max_tokens=5).lower())
    return rel / min(k, len(contexts))

TEST_CASES = [
    {'question': 'Does Pepperoni contain pork?'},
    {'question': 'What allergens are in Margherita?'},
    {'question': 'Can I order delivery Sunday north zone?'},
]
GEN_SYS = 'You are the assistant for Mamma Rosa Pizzeria. Answer ONLY using the provided context. Be concise.'
print('Running RAGAS-style evaluation...\n')
all_f, all_r, all_p = [], [], []
for tc in TEST_CASES:
    q = tc['question']
    contexts = [KB[cid] for cid in retrieve(q, k=3)]
    context  = '\n'.join(contexts)
    answer   = llm(GEN_SYS, f'Context:\n{context}\n\nQ: {q}', max_tokens=80)
    f = score_faithfulness(answer, context)
    r = score_answer_relevance(q, answer)
    p = score_context_precision(q, contexts)
    all_f.append(f); all_r.append(r); all_p.append(p)
    print(f'Q: {q}')
    print(f'   Answer: {answer[:90]}')
    print(f'   Faithfulness={f:.2f}  Relevance={r:.2f}  Precision={p:.2f}\n')
print(f'MEANS  Faith={np.mean(all_f):.3f}  Relev={np.mean(all_r):.3f}  Prec={np.mean(all_p):.3f}')

## 3 · LLM-as-Judge — Pairwise Preference

For open-ended tasks, use a judge model to compare two candidate answers. Swap A/B to detect position bias.

In [ ]:
JUDGE_SYS = 'You are a strict evaluator. Given question, context, and two answers (A/B), pick better on factual accuracy, completeness, conciseness. Write one sentence of reasoning then WINNER: A or WINNER: B or WINNER: TIE'
def judge(q, ctx, a, b):
    prompt = f'Context: {ctx}\n\nQuestion: {q}\n\nAnswer A: {a}\nAnswer B: {b}'
    raw = llm(JUDGE_SYS, prompt, max_tokens=100)
    m = re.search(r'WINNER:\s*(A|B|TIE)', raw, re.IGNORECASE)
    return m.group(1).upper() if m else 'UNKNOWN', raw.split('WINNER:')[0].strip()

q     = 'What is the cheapest gluten-free pizza?'
ctx   = KB['menu-01'] + ' ' + KB['menu-03']
good  = 'Margherita with GF base is 11.49 (9.99 + 1.50 GF upgrade).'
poor  = 'Yes we offer gluten-free options! Our pizzas are delicious and you can enjoy them worry-free!'

w1, r1 = judge(q, ctx, good, poor)
print(f'Q: {q}')
print(f'A: {good}')
print(f'B: {poor}')
print(f'\nReasoning: {r1}')
print(f'Winner: {w1}')

# Position-bias check
w2, _ = judge(q, ctx, poor, good)
remapped = 'B' if w2=='A' else ('A' if w2=='B' else 'TIE')
print(f'\nSwapped (bias check) remapped winner: {remapped}')
print('Consistent' if w1==remapped else 'Position bias detected!')

## 4 · Regression Test Harness

In [ ]:
THRESHOLDS = {'faithfulness': 0.70, 'answer_relevance': 0.60, 'context_precision': 0.60}
current    = {'faithfulness': float(np.mean(all_f)), 'answer_relevance': float(np.mean(all_r)), 'context_precision': float(np.mean(all_p))}
print('Regression Test Results\n')
print(f'{"Metric":<25} {"Score":>8} {"Threshold":>10} {"Status":>10}')
print('-'*57)
all_pass = True
for metric, thr in THRESHOLDS.items():
    s = current[metric]; passed = s >= thr; all_pass = all_pass and passed
    print(f'{metric:<25} {s:>8.3f} {thr:>10.3f} {"PASS" if passed else "FAIL"}  {"v" if passed else "X"}')
print('\nOVERALL:', 'All tests passed v' if all_pass else 'Some tests FAILED X')

## Summary

| Level | Technique | What it catches |
|---|---|---|
| Component | Retrieval P@k / R@k | Weak retriever |
| Pipeline | RAGAS faithfulness/relevance/precision | Hallucination, off-topic |
| System | LLM-as-judge pairwise | Open-ended quality regressions |
| CI/CD | Regression thresholds | Silent quality drops |

**Next:** [FineTuning/notebook.ipynb](../FineTuning/notebook.ipynb)